# APKGC biến thể A (mã công bố), seed 1234, MKG-W — huấn luyện rồi đo độ bền

Lặp lại đúng giao thức của biến thể A seed 42 (notebook `apkgc-three-branch`): siêu tham số đọc từ `scripts/run_base.sh`, nhiễu bật, phạt theo mã công bố. Khác duy nhất: **trần 2000 epoch** (thay cho 8000) để vừa quota 6 giờ GPU; seed 42 đã dừng sớm trước mốc này.

Sau huấn luyện, notebook đo độ bền của checkpoint tốt nhất bằng `robustness.py` (cùng công cụ với kernel `mmkgc-robustness-eval`); điều kiện sạch phải cho lại đúng MRR vừa đo.

Bật GPU T4 và Internet.

## 1. Môi trường

In [ ]:
import torch, subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
print("torch:", torch.__version__)

GPU_OK = False
NEED_DOWNGRADE = False

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    sm = f"sm_{cap[0]}{cap[1]}"
    print(f"GPU: {torch.cuda.get_device_name()}  {sm}")
    print("torch ho tro:", torch.cuda.get_arch_list())

    # Doc phien ban thoi khong du: phai thu chay that mot phep nhan ma tran.
    try:
        _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
        GPU_OK = True
        print("\nGPU chay duoc.")
    except Exception as e:
        print(f"\nGPU khong chay duoc: {type(e).__name__}")
        print(f"torch {torch.__version__} khong co kernel cho {sm}.")
        NEED_DOWNGRADE = True
else:
    print("Khong thay GPU.")

print()
if NEED_DOWNGRADE:
    print("=> O tiep theo se cai ban torch co ho tro GPU nay.")
elif not GPU_OK:
    print("=> Se chay bang CPU, cham hon dang ke.")


### Xử lý khi GPU không tương thích

Kaggle có thể cấp Tesla P100 (sm_60) trong khi bản PyTorch cài sẵn chỉ hỗ trợ từ sm_70. Ô dưới chỉ chạy khi gặp trường hợp đó.

In [ ]:
# Chi chay khi o tren phat hien GPU khong tuong thich.
#
# Kaggle doi khi cap Tesla P100 (sm_60) trong khi ban torch cai san chi ho tro
# tu sm_70 tro len. Ha ve ban torch co kernel cho sm_60.
#
# Khong can khoi dong lai kernel: phan huan luyen duoc goi qua subprocess nen
# tien trinh con se nap ban torch moi cai.

if NEED_DOWNGRADE:
    print("Dang cai torch 2.4.1+cu121 (co kernel cho sm_60)...")
    # Go torchvision/torchaudio truoc de tranh xung dot rang buoc phien ban
    !pip uninstall -y -q torchvision torchaudio 2>&1 | tail -1
    !pip install -q --force-reinstall torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

    import subprocess, sys
    probe = (
        "import torch;"
        "a=torch.randn(8,8,device='cuda');"
        "print('torch',torch.__version__,'|',torch.cuda.get_device_name());"
        "print('matmul OK', float((a@a).sum()))"
    )
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    print(r.stdout or r.stderr[-500:])
    GPU_OK = "matmul OK" in r.stdout
    print("\nTien trinh con dung duoc GPU:", GPU_OK)
else:
    print("Khong can thay doi torch.")


## 2. Tải mã nguồn

In [ ]:
%cd /kaggle/working
!rm -rf APKGC _openke
!git clone -q https://github.com/HubuKG/APKGC.git
!ls APKGC

## 3. Chuẩn bị mã nguồn

Ô dưới xử lý năm vấn đề khiến repo không chạy được ngoài môi trường tác giả:

1. **Thiếu mã C++.** Repo chỉ có `Base.so` biên dịch sẵn cho Linux x86-64; thư mục `mmkgc/base/` chứa `Base.cpp` không được commit. `mmkgc/` là bản sửa của OpenKE nên lấy phần này từ đó.
2. Gọi `.cuda()` cứng ở 17 vị trí.
3. `requirement.txt` thiếu `scikit-learn`, `scipy`.
4. Kết quả ghi ra `../../data/` — ngoài thư mục dự án.
5. Sập khi chạy dưới 100 epoch vì `best_model_wts` được truy cập vô điều kiện.

In [ ]:
import os, glob, subprocess, sys

A = "/kaggle/working/APKGC"

# ---- 1. Bo sung ma nguon C++ con thieu -------------------------------------
# Repo APKGC chi commit ban Base.so bien dich san cho Linux x86-64, con thu muc
# mmkgc/base/ chua Base.cpp thi khong duoc commit -> khong the bien dich lai.
# mmkgc/ von la ban sua doi cua OpenKE (THUNLP) nen lay base/ tu do.
if not os.path.exists(f"{A}/mmkgc/base/Base.cpp"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/thunlp/OpenKE.git",
                    "/kaggle/working/_openke"], check=True)
    subprocess.run(["cp", "-r", "/kaggle/working/_openke/openke/base",
                    f"{A}/mmkgc/base"], check=True)
    print("Da bo sung mmkgc/base/ tu OpenKE")

# ---- 2. Module chon thiet bi ----------------------------------------------
DEVICE_SRC = "\n".join([
    "import os",
    "import torch",
    '_f = os.environ.get("MMKGC_DEVICE", "").strip().lower()',
    "if _f:",
    "    DEVICE = torch.device(_f)",
    "elif torch.cuda.is_available():",
    '    DEVICE = torch.device("cuda")',
    "else:",
    '    DEVICE = torch.device("cpu")',
    "def device_name():",
    "    return str(DEVICE)",
    "",
])
with open(f"{A}/mmkgc/device.py", "w") as f:
    f.write(DEVICE_SRC)

# ---- 3. Thay .cuda() cung bang thiet bi dong ------------------------------
total = 0
for p in glob.glob(f"{A}/**/*.py", recursive=True):
    if p.endswith("device.py"):
        continue
    s = open(p, encoding="utf-8").read()
    if ".cuda()" not in s:
        continue
    n = s.count(".cuda()")
    rel = os.path.relpath(p, A)
    if rel.startswith("mmkgc/"):
        depth = os.path.relpath(p, f"{A}/mmkgc").count(os.sep)
        imp = "from " + "." * (depth + 1) + "device import DEVICE"
    else:
        imp = "from mmkgc.device import DEVICE"
    if "device import DEVICE" not in s:
        lines = s.split("\n")
        last = max((i for i, l in enumerate(lines)
                    if l.startswith(("import ", "from "))), default=0)
        lines.insert(last + 1, imp)
        s = "\n".join(lines)
    s = s.replace(".cuda()", ".to(DEVICE)")
    open(p, "w", encoding="utf-8").write(s)
    total += n
    print(f"  {rel:<44} {n}")
print(f"Da thay {total} cho .cuda()")

# ---- 4. Sua cac loi khien khong chay duoc ---------------------------------
p = f"{A}/mmkgc/config/Trainer.py"
s = open(p, encoding="utf-8").read()
s = s.replace("data_loader=self.test_dataloader, use_gpu=True)",
              "data_loader=self.test_dataloader, use_gpu=(DEVICE.type != 'cpu'))")
if "self.best_model_wts = None" not in s:
    s = s.replace("        self.Loss_log = Loss_log(early_stop=args.early_stop)",
                  "        self.Loss_log = Loss_log(early_stop=args.early_stop)\n"
                  "        self.best_model_wts = None")
open(p, "w", encoding="utf-8").write(s)

p = f"{A}/args.py"
s = open(p, encoding="utf-8").read()
if "save_steps" not in s:
    s = s.replace("    arg.add_argument('-num_proj', type=int, default=1)",
                  "    arg.add_argument('-num_proj', type=int, default=1)\n"
                  "    arg.add_argument('-save_steps', type=int, default=100)")
open(p, "w", encoding="utf-8").write(s)

p = f"{A}/run_KGC_train.py"
s = open(p, encoding="utf-8").read()
s = s.replace("    torch.cuda.manual_seed_all(args.seed)",
              "    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)")
s = s.replace("    data_root = osp.abspath(osp.join(this_dir, '..', '..', 'data', ''))",
              "    data_root = osp.abspath(osp.join(this_dir, 'output'))")
s = s.replace("        save_steps=100,", "        save_steps=args.save_steps,")
s = s.replace("use_gpu=True", "use_gpu=(DEVICE.type != 'cpu')")
if "from mmkgc.device import DEVICE" not in s:
    s = s.replace("from args import get_args",
                  "from args import get_args\nfrom mmkgc.device import DEVICE, device_name")

OLD = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if not osp.exists(save_dir):",
    '        torch.save(trainer.best_model_wts, f"{args.save}-MRR{trainer.Loss_log.get_acc()}")',
    "",
    "    kge_score.load_checkpoint(save_dir)",
])
NEW = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if trainer.best_model_wts is not None:",
    "        if not osp.exists(save_dir):",
    "            torch.save(trainer.best_model_wts, save_dir)",
    "        kge_score.load_checkpoint(save_dir)",
])
if OLD in s:
    s = s.replace(OLD, NEW)
    print("Da sua phan luu checkpoint")
open(p, "w", encoding="utf-8").write(s)
print("Da sua Trainer/args/entrypoint")


### Biên dịch thư viện C++

In [ ]:
%cd /kaggle/working/APKGC/mmkgc
!bash make.sh 2>&1 | tail -3
!file release/Base.so
%cd /kaggle/working

### Cài thư viện phụ thuộc

In [ ]:
!pip install -q transformers easydict unidecode tensorboard scikit-learn scipy 2>&1 | tail -2
import transformers
from transformers.activations import ACT2FN
from transformers.pytorch_utils import apply_chunking_to_forward
print('transformers', transformers.__version__, '— cac ham can thiet OK')

## 4. Tải bộ nhúng

Đặc trưng thị giác và văn bản đã trích sẵn, không nằm trong repo.

Nguồn: https://drive.google.com/drive/folders/1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB

In [ ]:
!pip install -q gdown
%cd /kaggle/working/APKGC
!gdown --folder 1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB -O ./emb_dl -q

import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
moved = 0
for f in pathlib.Path('emb_dl').rglob('*'):
    if f.suffix in ('.pth', '.json'):
        dst = pathlib.Path('embeddings') / f.name
        if not dst.exists():
            shutil.move(str(f), str(dst)); moved += 1
print('Da chuyen', moved, 'file')
!ls -la embeddings/

### Kiểm chứng số chiều

Bước này cũng là một quan sát cho nghiên cứu: hai bộ dữ liệu dùng encoder khác nhau nên số chiều thị giác chênh nhau hơn mười lần.

In [ ]:
# Doc so chieu qua subprocess: tien trinh chinh co the dang giu ban torch cu
# trong bo nho sau khi ha phien ban, nen torch.load() o day se hong.
import subprocess, sys

probe = """
import torch, os
print(f"{'file':<24}{'shape':>18}")
print('-' * 42)
for f in sorted(os.listdir('embeddings')):
    if f.endswith('.pth'):
        t = torch.load(f'embeddings/{f}', map_location='cpu', weights_only=False)
        print(f"{f:<24}{str(tuple(t.shape)):>18}")
"""
r = subprocess.run([sys.executable, "-c", probe],
                   cwd="/kaggle/working/APKGC", capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])


## 5. Bộ khung chạy thí nghiệm

In [ ]:
import subprocess, sys, os, re, json, time, statistics

A = "/kaggle/working/APKGC"
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)

# APKGC in ket qua qua logger:
#   mrr:0.3742,  mr:145.2,  hit10:0.5011, hit3:0.4038, hit1:0.3061
LOGGER_LINE = re.compile(
    r"mrr:\s*([\d.]+).*?mr:\s*([\d.]+).*?hit10:\s*([\d.]+)"
    r".*?hit3:\s*([\d.]+).*?hit1:\s*([\d.]+)")

# Du phong: dinh dang printf cua mmkgc/base/Test.h
METRIC_LINE = re.compile(
    r"^\s*(l|r|averaged)\(filter\):\s*"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)", re.MULTILINE)

SHELL_VAR = re.compile(r"^\s*([A-Z_][A-Z0-9_]*)=(\S+)\s*$", re.MULTILINE)


def read_author_config():
    """Doc sieu tham so tu scripts/run_base.sh — file ma run.sh thuc su goi."""
    path = f"{A}/scripts/run_base.sh"
    cfg = dict(SHELL_VAR.findall(open(path).read()))
    for k in ["EMB_DIM", "NUM_BATCH", "MARGIN", "LR", "NEG_NUM", "EPOCH"]:
        assert k in cfg, f"{path} thieu {k}"
    return cfg


def parse_metrics(text):
    hits = LOGGER_LINE.findall(text)
    if hits:
        m = hits[-1]
        return dict(mrr=float(m[0]), mr=float(m[1]),
                    hit10=float(m[2]), hit3=float(m[3]), hit1=float(m[4]))
    sides = {}
    for m in METRIC_LINE.finditer(text):
        sides[m.group(1)] = dict(
            mrr=float(m.group(2)), mr=float(m.group(3)),
            hit10=float(m.group(4)), hit3=float(m.group(5)),
            hit1=float(m.group(6)))
    if "averaged" in sides:
        return sides["averaged"]
    if "l" in sides and "r" in sides:
        return {k: (sides["l"][k] + sides["r"][k]) / 2
                for k in ("mrr", "mr", "hit10", "hit3", "hit1")}
    return None


# Hai tap du lieu dung chien luoc hop nhat khac nhau, theo dung run.sh goc:
#   dong 1: DB15K  num_proj=2  Mformer_hd_graph
#   dong 2: MKG-W  num_proj=1  Mformer_hd_mean
DATASET_ARGS = {
    "DB15K": dict(num_proj=2, joint_way="Mformer_hd_graph"),
    "MKG-W": dict(num_proj=1, joint_way="Mformer_hd_mean"),
}


def run_seed(dataset, seed, cfg, epoch=None, save_steps=100):
    n_epoch = epoch or int(cfg["EPOCH"])
    da = DATASET_ARGS[dataset]
    os.makedirs(f"{A}/output/mmkgc/checkpoint", exist_ok=True)

    cmd = [sys.executable, "run_KGC_train.py",
           f"-dataset={dataset}",
           f"-num_proj={da['num_proj']}",
           "-use_intermediate=0",
           f"-joint_way={da['joint_way']}",
           f"-batch_size={cfg['NUM_BATCH']}",
           f"-margin={cfg['MARGIN']}",
           f"-epoch={n_epoch}",
           f"-dim={cfg['EMB_DIM']}",
           f"-neg_num={cfg['NEG_NUM']}",
           f"-learning_rate={cfg['LR']}",
           f"-noise_update={cfg.get('NU', 'epoch')}",
           "-noise_ratio=0.2", "-mask_ratio=0.7",
           f"-use_pool={cfg.get('POOL', 1)}",
           f"-add_noise={cfg.get('NOISE', 1)}",
           "-num_hidden_layers=1", "-num_attention_heads=2",
           f"-seed={seed}", f"-save=seed{seed}",
           f"-save_steps={save_steps}", f"-exp_id=seed{seed}"]

    print(f"  seed {seed} ...", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, cwd=A, capture_output=True, text=True)
    dt = time.time() - t0

    log = f"{OUT}/{dataset}_seed{seed}.log"
    with open(log, "w") as f:
        f.write(r.stdout + "\n===STDERR===\n" + r.stderr)

    m = parse_metrics(r.stdout + r.stderr)
    if m is None:
        print(f"    that bai sau {dt/60:.1f} phut — xem {log}")
        print("    " + "\n    ".join((r.stdout + r.stderr).strip().splitlines()[-10:]))
        return None
    m["seed"] = seed
    m["seconds"] = round(dt, 1)
    print(f"    MRR={m['mrr']:.4f}  H@1={m['hit1']:.4f}  ({dt/60:.1f} phut)")
    return m


def summarize(runs, dataset, cfg, n_epoch):
    if len(runs) < 2:
        print("Khong du ket qua de tinh do phan tan.")
        return None

    print("\n" + "=" * 68)
    print(f"KET QUA — APKGC tren {dataset}, {len(runs)} seed")
    print("=" * 68)
    print(f"{'seed':>6} {'MRR':>9} {'H@1':>9} {'H@3':>9} {'H@10':>9} {'phut':>7}")
    for r in runs:
        print(f"{r['seed']:>6} {r['mrr']:>9.4f} {r['hit1']:>9.4f} "
              f"{r['hit3']:>9.4f} {r['hit10']:>9.4f} {r['seconds']/60:>7.1f}")

    print()
    stats = {}
    for k in ("mrr", "hit1", "hit3", "hit10"):
        v = [r[k] for r in runs]
        stats[k] = dict(mean=statistics.mean(v), std=statistics.stdev(v),
                        lo=min(v), hi=max(v), spread=max(v) - min(v))
        s = stats[k]
        print(f"{k.upper():>6}: {s['mean']:.4f} +/- {s['std']:.4f}"
              f"   [{s['lo']:.4f}, {s['hi']:.4f}]   bien do {s['spread']:.4f}")

    payload = dict(model="APKGC", dataset=dataset, epoch=n_epoch,
                   config_source="scripts/run_base.sh", config=cfg,
                   dataset_args=DATASET_ARGS[dataset], runs=runs, stats=stats)
    with open(f"{OUT}/{dataset}_summary.json", "w") as f:
        json.dump(payload, f, indent=2)
    print(f"\nDa luu {OUT}/{dataset}_summary.json")
    return stats


In [ ]:
# Moi con so duoi day doc truc tiep tu bang trong bai bao APKGC (AAAI 2025).
PUBLISHED = {
    "DB15K": dict(mrr=0.364, hit1=0.282, hit3=0.413, hit10=0.527),
    "MKG-W": dict(mrr=0.374, hit1=0.306, hit3=0.404, hit10=0.501),
}

GAPS = [
    dict(dataset="DB15K", label="APKGC vs NativE", gap=0.002,
         src="Bang 2, tr.15010 (0.364 vs 0.362)"),
    dict(dataset="MKG-W", label="APKGC vs NativE", gap=0.008,
         src="Bang 2, tr.15010 (0.374 vs 0.366)"),
    dict(dataset="MKG-W", label="Dong gop attention penalty", gap=0.014,
         src="Bang 4, tr.15011 (0.374 vs 0.360)"),
    dict(dataset="MKG-W", label="Dong gop noise sampling", gap=0.016,
         src="Bang 4, tr.15011 (0.374 vs 0.358)"),
]


def interpret(stats, dataset):
    std = stats["mrr"]["std"]
    mean = stats["mrr"]["mean"]
    pub = PUBLISHED[dataset]["mrr"]

    print("=" * 68)
    print("DIEN GIAI")
    print("=" * 68)
    print(f"So cong bo trong bai bao : MRR = {pub:.4f}")
    print(f"So do lai duoc           : MRR = {mean:.4f} +/- {std:.4f}")
    print(f"Chenh lech               : {mean - pub:+.4f}")
    print()
    print("Doi chieu do phan tan voi khoang cach giua cac phuong phap:")
    for g in GAPS:
        if g["dataset"] != dataset:
            continue
        v = "NAM TRONG nhieu seed" if std >= g["gap"] else "vuot nhieu seed"
        print(f"  {g['label']:<34} {g['gap']:.4f}  ->  {v}")
        print(f"    nguon: APKGC AAAI 2025, {g['src']}")

    print()
    if std > 0.005:
        print("KET LUAN: do phan tan lon hon phan lon khoang cach duoc cong bo.")
        print("Luan diem cua nghien cuu dung vung — nen tien hanh day du.")
    elif std < 0.002:
        print("KET LUAN: mo hinh on dinh giua cac seed. Cac cai tien cong bo")
        print("nhieu kha nang la that. Nen chuyen trong tam sang phan audit")
        print("tham so va tinh khong dong nhat cua encoder.")
    else:
        print("KET LUAN: chua dut khoat. Can them seed hoac them mo hinh.")

    print()
    print("PHAM VI KET LUAN")
    print("Ket qua do do phan tan cua lan chay nay voi cau hinh duoc cong bo.")
    print("No KHONG cho phep suy ra tac gia da chay bao nhieu lan hay thu bao")
    print("nhieu cau hinh: kho ma chi phan anh nhung gi duoc cong bo. Phat bieu")
    print("kiem chung duoc la 'bai bao khong bao cao do lech chuan', khong phai")
    print("'tac gia khong do do lech chuan'.")


### Đối chiếu cấu hình: bài báo, README và mã nguồn

Ba nguồn này không khớp nhau. Ô dưới in ra để thấy rõ.

In [ ]:
cfg = read_author_config()
print('Doc tu scripts/run_base.sh (file ma run.sh thuc su goi):')
for k, v in cfg.items():
    print(f'  {k:<12} = {v}')
print()
print('Bai bao (AAAI 2025, tr.15010, muc Parameter Settings):')
print('  dim=128, batch=2048, neg=64, lr=1e-4')
print()
print('README cua repo:')
print('  EMB_DIM=128, NUM_BATCH=2048, NEG_NUM=64  (khop bai bao)')
print()
print('=> batch va neg trong script duoc goi chi bang MOT NUA gia tri cong bo.')
print('   Thi nghiem nay dung dung file ma run.sh goi.')

In [ ]:
%%writefile /kaggle/working/robustness.py
#!/usr/bin/env python3
"""Đánh giá độ bền của checkpoint MMKGC (APKGC, NativE) khi phương thức bị hỏng lúc test.

VÌ SAO. Bài APKGC nói lấy mẫu nhiễu làm mô hình "bền vững" hơn, nhưng chỉ đánh
giá trên tập test sạch. Công cụ này làm hỏng đặc trưng ảnh/văn bản lúc test rồi
đo lại MRR, Hits@K, không huấn luyện lại. NativE được đo cùng cách để so sánh.

CÁCH TÍNH NHANH MÀ VẪN ĐÚNG. Vector hợp nhất của một thực thể chỉ phụ thuộc vào
ba vector của chính nó (APKGC), hoặc thêm cổng của quan hệ (NativE: `rel_gate`).
Nên tính vector hợp nhất cho mọi thực thể một lần (APKGC) hoặc một lần cho mỗi
quan hệ (NativE), rồi chấm RotatE cho mọi truy vấn bằng phép toán ma trận. Xếp
hạng theo đúng `mmkgc/base/Test.h` của cả hai repo: đếm số ứng viên có khoảng
cách NHỎ HƠN HẲN đáp án, bỏ qua bộ ba đã biết (train + valid + test), hạng =
đếm + 1; MRR, Hits@K trung bình trên hai chiều đầu/đuôi. Điều kiện sạch phải
cho lại đúng MRR của checkpoint — tham số `--expect` kiểm chuyện đó.

CÁC KIỂU LÀM HỎNG (áp lên bảng đặc trưng đóng băng `img_embeddings`,
`text_embeddings`; vector cấu trúc không đổi):
  missing  thay đặc trưng bằng vector trung bình của các thực thể có phương thức
           đó — đúng cách mã APKGC tự điền cho thực thể thiếu phương thức
  shuffle  hoán vị đặc trưng giữa các thực thể được chọn
  noise_σ  cộng nhiễu Gauss σ × std theo từng chiều
mỗi kiểu với tỉ lệ thực thể bị hỏng p ∈ {0,5; 1}. Kiểu ngẫu nhiên lặp nhiều seed.

Dùng (mỗi mô hình một tiến trình vì hai repo cùng tên gói `mmkgc`):
    python apkgc_robustness.py --model apkgc --repo <APKGC> --ckpt <ckpt> \\
        --missing <missing.json> --expect 0.3629 --out <json>
    python apkgc_robustness.py --model native --repo <NATIVE> --ckpt <ckpt> \\
        --missing <missing.json> --expect 0.3684 --out <json>
"""
import argparse
import json
import os
import sys
import time
from collections import defaultdict

import torch

# Cờ dựng APKGC: giống lệnh huấn luyện của biến thể A (log seed42).
APKGC_ARGV = ["-dataset=MKG-W", "-num_proj=1", "-use_intermediate=0",
              "-joint_way=Mformer_hd_mean", "-dim=128", "-margin=12",
              "-noise_update=epoch", "-noise_ratio=0.2", "-mask_ratio=0.7",
              "-use_pool=1", "-add_noise=1", "-num_hidden_layers=1",
              "-num_attention_heads=2"]
# NativE trên MKG-W: scripts/run_mkgw.sh của tác giả.
NATIVE_DIM, NATIVE_MARGIN = 250, 4.0


def doc_bo_ba(path):
    with open(path) as f:
        n = int(f.readline())
        out = [tuple(map(int, f.readline().split())) for _ in range(n)]
    return [(h, r, t) for h, t, r in out]          # tệp OpenKE ghi h t r


def dung_mo_hinh(kind, repo, ckpt, device):
    """Dựng mô hình bằng mã của chính repo, nạp checkpoint; đặc trưng lấy từ checkpoint."""
    sd = torch.load(ckpt, map_location="cpu", weights_only=False)
    # NativE lưu cả bộ bọc NegativeSamplingGP: khoá mô hình có tiền tố "model.",
    # còn "loss.*" và hằng số ngoài cùng không thuộc mô hình.
    if any(k.startswith("model.") for k in sd):
        sd = {k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}
    img, txt = sd["img_embeddings.weight"], sd["text_embeddings.weight"]
    n_ent, n_rel = sd["ent_embeddings.weight"].shape[0], sd["rel_embeddings.weight"].shape[0]
    sys.path.insert(0, repo)
    cwd = os.getcwd()
    os.chdir(repo)                                    # APKGC đọc ./embeddings/missing.json
    try:
        if kind == "apkgc":
            from args import get_args
            from mmkgc.module.model.APKGC import APKGC
            sys.argv = ["x"] + APKGC_ARGV
            args = get_args()
            m = APKGC(args=args, ent_tot=n_ent, rel_tot=n_rel, dim=args.dim,
                      margin=args.margin, epsilon=2.0, img_emb=img.clone(), text_emb=txt.clone())
        else:
            from mmkgc.module.model import AdvRelRotatE
            m = AdvRelRotatE(ent_tot=n_ent, rel_tot=n_rel, dim=NATIVE_DIM,
                             margin=NATIVE_MARGIN, epsilon=2.0, img_emb=img.clone(), text_emb=txt.clone())
    finally:
        os.chdir(cwd)
    thieu, thua = m.load_state_dict(sd, strict=False)
    # APKGC: hai bảng nhiễu tạo trong update_noise(), không tồn tại lúc dựng.
    assert not thieu, f"checkpoint thiếu khoá: {thieu}"
    assert set(thua) <= {"text_embeddings_noise.weight", "img_embeddings_noise.weight"}, thua
    return m.to(device).eval()


@torch.no_grad()
def vector_hop_nhat(kind, m, img_w, txt_w, rel, device, lo=4096):
    """Vector hợp nhất của mọi thực thể (với NativE: cho quan hệ `rel`)."""
    n = m.ent_embeddings.weight.shape[0]
    ra = []
    for s in range(0, n, lo):
        idx = torch.arange(s, min(n, s + lo), device=device)
        es = m.ent_embeddings(idx)
        ev = m.img_proj(img_w[idx])
        et = m.text_proj(txt_w[idx])
        if kind == "apkgc":
            ra.append(m.get_joint_embeddings(es, ev, et))
        else:
            rg = m.rel_gate(torch.full_like(idx, rel))
            ra.append(m.get_joint_embeddings(es, ev, et, rg))
    return torch.cat(ra)


@torch.no_grad()
def xep_hang(kind, m, img_w, txt_w, test, da_biet_t, da_biet_h, device, lo=16):
    """Hạng lọc của đáp án cho cả hai chiều, theo quy tắc của Test.h."""
    pi = m.pi_const.item()
    rng = m.rel_embedding_range.item()
    theo_qh = defaultdict(list)
    for x in test:
        theo_qh[x[1]].append(x)
    J_chung = vector_hop_nhat(kind, m, img_w, txt_w, None, device) if kind == "apkgc" else None
    hang = []
    for rel, ds in theo_qh.items():
        J = J_chung if J_chung is not None else vector_hop_nhat(kind, m, img_w, txt_w, rel, device)
        re_all, im_all = torch.chunk(J, 2, dim=-1)
        for s in range(0, len(ds), lo):
            lo_bt = ds[s:s + lo]
            h = torch.tensor([x[0] for x in lo_bt], device=device)
            r = torch.tensor([x[1] for x in lo_bt], device=device)
            t = torch.tensor([x[2] for x in lo_bt], device=device)
            ph = m.rel_embeddings(r) / (rng / pi)
            cr, sr = torch.cos(ph), torch.sin(ph)
            rh, ih = re_all[h], im_all[h]
            rt, it = re_all[t], im_all[t]
            # đuôi: |h∘r − t'| ; đầu: |conj(r)∘t − h'|
            for qr, qi, dung, loai in (
                    (rh * cr - ih * sr, rh * sr + ih * cr, t,
                     [da_biet_t[(a, rel)] for a, _, _ in lo_bt]),
                    (cr * rt + sr * it, cr * it - sr * rt, h,
                     [da_biet_h[(rel, c)] for _, _, c in lo_bt])):
                d = torch.sqrt((qr[:, None, :] - re_all[None]) ** 2 +
                               (qi[:, None, :] - im_all[None]) ** 2).sum(-1)     # [B, N]
                hon = d < d.gather(1, dung[:, None])
                for i, ung_vien in enumerate(loai):      # bỏ ứng viên tạo thành bộ ba đã biết
                    hon[i, list(ung_vien)] = False
                hang.extend((hon.sum(1) + 1).tolist())
    return hang


def chi_so(hang):
    n = len(hang)
    return dict(mrr=sum(1 / x for x in hang) / n, mr=sum(hang) / n,
                hit1=sum(x <= 1 for x in hang) / n, hit3=sum(x <= 3 for x in hang) / n,
                hit10=sum(x <= 10 for x in hang) / n)


def lam_hong(w, kieu, p, sd, mean):
    w = w.clone()
    g = torch.Generator().manual_seed(sd)
    n = w.shape[0]
    chon = torch.randperm(n, generator=g)[: int(round(p * n))].to(w.device)
    if kieu == "missing":
        w[chon] = mean
    elif kieu == "shuffle":
        w[chon] = w[chon[torch.randperm(len(chon), generator=g).to(w.device)]]
    elif kieu.startswith("noise"):
        sigma = float(kieu.split("_")[1])
        std = w.std(0, keepdim=True)
        w[chon] = w[chon] + sigma * std * torch.randn(len(chon), w.shape[1], generator=g).to(w.device)
    return w


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["apkgc", "native"], required=True)
    ap.add_argument("--repo", required=True, help="thư mục repo của mô hình (chứa mmkgc/ và benchmarks/)")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--missing", required=True, help="missing.json: chỉ số thực thể thiếu ảnh/văn bản")
    ap.add_argument("--expect", type=float, help="MRR sạch phải khớp (sai số 5e-4)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--seeds", type=int, default=3, help="số seed cho kiểu làm hỏng ngẫu nhiên")
    ap.add_argument("--budget", type=float, default=2.5, help="giờ tối đa; quá thì dừng, kết quả đã ghi giữ nguyên")
    a = ap.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t_bat_dau = time.time()

    m = dung_mo_hinh(a.model, a.repo, a.ckpt, device)
    bm = os.path.join(a.repo, "benchmarks/MKG-W")
    test = doc_bo_ba(f"{bm}/test2id.txt")
    da_biet_t, da_biet_h = defaultdict(set), defaultdict(set)
    for f in ("train2id.txt", "valid2id.txt", "test2id.txt"):
        for h, r, t in doc_bo_ba(f"{bm}/{f}"):
            da_biet_t[(h, r)].add(t)
            da_biet_h[(r, t)].add(h)

    img0, txt0 = m.img_embeddings.weight.data, m.text_embeddings.weight.data
    ms = json.load(open(a.missing))
    thieu_img = set(ms["MKG-W-visual.pth"]); thieu_txt = set(ms["MKG-W-textual.pth"])
    co_img = torch.tensor([i for i in range(img0.shape[0]) if i not in thieu_img], device=device)
    co_txt = torch.tensor([i for i in range(txt0.shape[0]) if i not in thieu_txt], device=device)
    img_mean, txt_mean = img0[co_img].mean(0), txt0[co_txt].mean(0)

    def danh_gia(img_w, txt_w):
        t0 = time.time()
        kq = chi_so(xep_hang(a.model, m, img_w, txt_w, test, da_biet_t, da_biet_h, device))
        kq["giay"] = round(time.time() - t0, 1)
        return kq

    sach = danh_gia(img0, txt0)
    print(f"[{a.model}] sạch:", {k: round(v, 4) for k, v in sach.items()}, flush=True)
    ket_qua = {"model": a.model, "checkpoint": os.path.basename(a.ckpt), "dataset": "MKG-W",
               "sach": sach, "kiem_chung_mrr": a.expect, "dieu_kien": []}
    if a.expect is not None:
        ket_qua["kiem_chung_dat"] = abs(sach["mrr"] - a.expect) < 5e-4
        assert ket_qua["kiem_chung_dat"], f"MRR sạch {sach['mrr']:.4f} lệch {a.expect}: bộ đánh giá sai"
    json.dump(ket_qua, open(a.out, "w", encoding="utf-8"), indent=1, ensure_ascii=False)

    for kieu in ("missing", "shuffle", "noise_0.5", "noise_1", "noise_2"):
        for p in (1.0, 0.5):
            for pt in ("img", "txt", "both"):
                if time.time() - t_bat_dau > a.budget * 3600:
                    print("hết ngân sách thời gian, dừng", flush=True)
                    return
                tat_dinh = kieu == "missing" and p == 1.0
                lan = []
                for sd in range(1 if tat_dinh else a.seeds):
                    img_w = lam_hong(img0, kieu, p, sd, img_mean) if pt in ("img", "both") else img0
                    txt_w = lam_hong(txt0, kieu, p, 1000 + sd, txt_mean) if pt in ("txt", "both") else txt0
                    lan.append(danh_gia(img_w, txt_w))
                tb = {k: sum(x[k] for x in lan) / len(lan) for k in ("mrr", "hit1", "hit3", "hit10")}
                if len(lan) > 1:
                    mu = tb["mrr"]
                    tb["mrr_std"] = (sum((x["mrr"] - mu) ** 2 for x in lan) / (len(lan) - 1)) ** 0.5
                tb.update(kieu=kieu, ti_le=p, phuong_thuc=pt, so_seed=len(lan),
                          delta_mrr=tb["mrr"] - sach["mrr"], giay=sum(x["giay"] for x in lan))
                ket_qua["dieu_kien"].append(tb)
                print(f"[{a.model}] {kieu:10s} p={p:<4} {pt:5s} MRR={tb['mrr']:.4f} "
                      f"Δ={tb['delta_mrr']:+.4f} ({tb['giay']:.0f}s)", flush=True)
                json.dump(ket_qua, open(a.out, "w", encoding="utf-8"), indent=1, ensure_ascii=False)


if __name__ == "__main__":
    main()


In [ ]:
# ── Huấn luyện biến thể A, seed 1234 ──────────────────────────────
SEED = 1234
r = run_seed("MKG-W", SEED, cfg, epoch=2000, save_steps=100)
if r:
    r.update(bien_the="A_goc", epoch_cap=2000, config_source="scripts/run_base.sh")
    json.dump(r, open(f"{OUT}/A_goc_seed{SEED}.json", "w"), indent=2)
print(r)

In [ ]:
# ── Đo độ bền trên checkpoint tốt nhất ───────────────────────────────
import glob
ck = sorted(glob.glob(f"{A}/output/mmkgc/checkpoint/seed{SEED}-MRR*"))
print(ck)
if r and ck:
    cmd = [sys.executable, "/kaggle/working/robustness.py", "--model", "apkgc", "--repo", A,
           "--ckpt", ck[-1], "--missing", f"{A}/embeddings/missing.json",
           "--expect", str(round(r["mrr"], 4)), "--out", f"{OUT}/robustness_apkgc_A_seed{SEED}.json",
           "--budget", "0.5"]
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-6000:]); print(p.stderr[-3000:] if p.returncode else "")